## 🌊 Flow Matching with Contextual U-Net

Welcome to this notebook on **Flow Matching** — a modern, elegant approach to generative modelling.
Instead of learning to reverse a complex noising chain, flow matching learns a **vector field** that
transports samples from a simple source distribution (standard Gaussian) to the data distribution along
straight paths.

We apply this to a conditional image generation task on the CLEVR dataset, using a Contextual U-Net backbone.

### 🧠 What you'll learn:
- The core idea behind **Conditional Flow Matching (CFM)**
- How to define a **linear interpolation path** between noise and data
- How to train a network to predict the **velocity field** along that path
- How to generate samples via **ODE integration** (Euler method)
- How context conditioning works in the U-Net architecture

### 📖 Key concepts

| | Flow Matching | Direct Regression (baseline) |
|---|---|---|
| **Input to network** | Interpolated image $x_t = (1-t)x_0 + t\epsilon$ | All-ones image |
| **Time input** | Sampled $t \sim U(0,1)$ | No timesteps |
| **Target** | Velocity $v^* = \epsilon - x_0$ | Direct $x_0$ |
| **Sampling** | Euler ODE integration over $n\_T$ steps | Single forward pass from ones |

### 📚 References
- Lipman et al., *Flow Matching for Generative Modelling* (2022): https://arxiv.org/abs/2210.02747
- Albergo & Vanden-Eijnden, *Building Normalizing Flows with Stochastic Interpolants* (2022)
- Liu et al., *Flow Straight and Fast: Rectified Flow* (2022)

In [ ]:
"""
Setup global paths for the project from environment variables.
"""
import os

# local
CV_PATH_FLOW_MATCHING = "./data/image_gen"
# tfpool
CV_PATH_FLOW_MATCHING = "/project/dl2026s/lmbcvtst/data/image_gen"
os.environ["HF_HOME"] = "/project/dl2026s/lmbcvtst/data/hf_cache"


In [ ]:
# Imports and Device Setup
import os
import json
import glob
import random
import math 
from tqdm import tqdm

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import torchvision.utils as vutils

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from argparse import Namespace
from einops import rearrange, pack, unpack

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

#from paths import CV_PATH_FLOW_MATCHING

print(CV_PATH_FLOW_MATCHING)

# Parameters (Override as needed for interactive runs)
These parameters define the behaviour of training, model architecture, dataset, and sampling.

| Parameter | Description |
|----------|-------------|
| `lrate` | Learning rate |
| `test_size` | Size of the object in the test dataset |
| `num_samples` | Total number of training samples per epoch |
| `batch_size` | Number of images per training batch |
| `n_T` | Number of ODE integration steps at **sampling** time |
| `n_feat` | Feature size in U-Net |
| `n_sample` | Number of samples generated during evaluation |
| `n_epoch` | Number of training epochs |
| `dataset` | Dataset name (e.g. CLEVR) |
| `pixel_size` | Image resolution (e.g. 28×28) |

> ⚡ **Note on `n_T`:** `n_T` controls the number of Euler integration steps at sampling time. Flow matching paths are near-linear, so 20–50 steps typically suffice for excellent results.

In [ ]:
import os as _os

# Using Namespace to emulate argparse arguments.
# The examiner can override checkpoint paths without editing the notebook:
#   FM_CKPT=results/flow_matching/flow_model.pth \
#   FM_BASELINE_CKPT=results/baseline/flow_model_baseline.pth \
#       python -m pytest test_notebook.py -v

# Output directories — one per model variant
_fm_dir       = _os.environ.get("FM_OUT_DIR",       "results/flow_matching")
_baseline_dir = _os.environ.get("FM_BASELINE_DIR",  "results/baseline")
_os.makedirs(_fm_dir,       exist_ok=True)
_os.makedirs(_baseline_dir, exist_ok=True)

args = Namespace(
    lrate=1e-4,
    test_size=1.6,
    num_samples=5000,
    batch_size=64,
    n_T=50,           # Number of Euler integration steps at sampling time
    n_feat=256,
    n_sample=64,
    n_epoch=50,
    type_attention="",
    pixel_size=28,
    dataset="clevr",
    seed=1,
    # Checkpoint paths default to inside the output subdirs
    save_model_path=_os.environ.get(
        "FM_CKPT", _os.path.join(_fm_dir, "flow_model.pth")),
    baseline_model_path=_os.environ.get(
        "FM_BASELINE_CKPT", _os.path.join(_baseline_dir, "flow_model_baseline.pth")),
    output_dir=_fm_dir,
    baseline_output_dir=_baseline_dir,
    load_model_path=None,
)
print(f"Flow matching output dir : {args.output_dir}")
print(f"Flow matching checkpoint : {args.save_model_path}")
print(f"Baseline output dir      : {args.baseline_output_dir}")
print(f"Baseline checkpoint      : {args.baseline_model_path}")


## 🧩 CLEVR Dataset: Custom Dataset Loader for Conditional Generation

This class loads a subset of the [CLEVR dataset](https://cs.stanford.edu/people/jcjohns/clevr/) and provides
contextual attributes (shape, colour, size) as conditioning signals.

### 🔧 Constructor: `__init__`

| Argument | Description |
|---|---|
| `transform` | Optional image transform (e.g. `transforms.Compose`) |
| `num_samples` | Number of samples returned per epoch |
| `dataset` | Dataset name (used in file path) |
| `configs` | Shape-colour-size combos to include |
| `training` | Train or test split |
| `test_size` | Optional size override for test sampling |

### 🖼️ `__getitem__`: Loading and Labelling an Image

Returns `(img, label)` where `label = {0: shape, 1: color_array, 2: size_array}`.

### 📏 `__len__`: Returns `num_samples`


In [ ]:
class CLEVR_dataset(Dataset):
    def __init__(self, transform=None, num_samples=5000, dataset="", configs="", training=True, test_size=None):
        self.training = training
        self.test_size = test_size
        self.dataset = dataset

        prefix = "CLEVR"
        ext = ".png"

        if training:
            self.train_image_paths = []
            for config in configs:
                new_paths = glob.glob(f"{CV_PATH_FLOW_MATCHING}/{dataset}/train/{prefix}_{config}_*{ext}")
                self.train_image_paths.extend(new_paths)
            self.len_data = len(self.train_image_paths)
        else:
            self.test_image_paths = glob.glob(f"{CV_PATH_FLOW_MATCHING}/{dataset}/test/{prefix}_{configs}_*{ext}")
            self.len_data = len(self.test_image_paths)

        self.num_samples = num_samples
        self.transform = transform

    def __getitem__(self, index):
        if self.training:
            ipath = random.randint(0, len(self.train_image_paths) - 1)
            img_path = self.train_image_paths[ipath]
        else:
            ipath = random.randint(0, len(self.test_image_paths) - 1)
            img_path = self.test_image_paths[ipath]

        img = Image.open(img_path).convert('RGB')
        if self.transform is not None:
            img = self.transform(img)

        name_labels = img_path.split("_")[-2]

        with open(img_path.replace(".png", ".json"), 'r') as f:
            my_dict = json.loads(f.read())
            _size = my_dict[0]
            _color = my_dict[1][:3]

        if self.training:
            size, color = _size, _color
        else:
            colors_map = {
                '0': [0.9, 0.1, 0.1],
                '1': [0.1, 0.1, 0.9],
                '2': [0.1, 0.9, 0.1]
            }
            size = 2.6 if int(name_labels[2]) == 0 else self.test_size
            color = colors_map[name_labels[1]]

        size = np.array(size, dtype=np.float32)
        color = np.array(color, dtype=np.float32)

        label = {0: int(name_labels[0]), 1: color, 2: size}
        return img, label

    def __len__(self):
        return self.num_samples


## 🧱 Model Components

The U-Net architecture receives:
- `x`: the interpolated image $x_t$ (mixture of data and Gaussian noise)
- `t`: the scalar time $t \in [0, 1]$ (continuous value)
- `c`: context labels (shape, colour, size)

And outputs a **velocity field** of the same shape as the input image.

### 🔁 Residual Blocks
Preserve information across layers and stabilise training.

### 🧩 U-Net Architecture
Encoder–bottleneck–decoder with skip connections and context/time embeddings.


In [ ]:
class ResidualConvBlock(nn.Module):
    def __init__(
        self, in_channels: int, out_channels: int, is_res: bool = False
    ) -> None:
        super().__init__()
        self.same_channels = (in_channels == out_channels)
        self.is_res = is_res
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, 1, 1),
            nn.BatchNorm2d(out_channels),
            nn.GELU(),
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, 3, 1, 1),
            nn.BatchNorm2d(out_channels),
            nn.GELU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.is_res:
            x1 = self.conv1(x)
            x2 = self.conv2(x1)
            if self.same_channels:
                out = x + x2
            else:
                out = x1 + x2
            return out
        else:
            x1 = self.conv1(x)
            x2 = self.conv2(x1)
            return x2


class UnetDown(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(UnetDown, self).__init__()
        self.model = nn.Sequential(
            ResidualConvBlock(in_channels, out_channels),
            nn.MaxPool2d(2)
        )

    def forward(self, x):
        return self.model(x)


class UnetUp(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(UnetUp, self).__init__()
        self.model = nn.Sequential(
            nn.ConvTranspose2d(in_channels, out_channels, 2, 2),
            ResidualConvBlock(out_channels, out_channels),
            ResidualConvBlock(out_channels, out_channels),
        )

    def forward(self, x, skip):
        x = torch.cat((x, skip), 1)
        x = self.model(x)
        return x


class EmbedFC(nn.Module):
    def __init__(self, input_dim, emb_dim):
        super(EmbedFC, self).__init__()
        self.input_dim = input_dim
        self.model = nn.Sequential(
            nn.Linear(input_dim, emb_dim),
            nn.GELU(),
            nn.Linear(emb_dim, emb_dim),
        )

    def forward(self, x):
        x_in = x.view(-1, self.input_dim)
        return self.model(x_in)


class ContextUnet(nn.Module):
    def __init__(self, in_channels, n_feat=256, n_classes=[2, 3, 1]):
        super(ContextUnet, self).__init__()

        self.in_channels = in_channels
        self.n_contexts = len(n_classes)
        self.n_feat = 2 * n_feat
        self.n_classes = n_classes

        self.init_conv = ResidualConvBlock(in_channels, n_feat, is_res=True)

        self.down1 = UnetDown(n_feat, n_feat)
        self.down2 = UnetDown(n_feat, 2 * n_feat)

        self.to_vec = nn.Sequential(nn.AvgPool2d(7), nn.GELU())

        self.timeembed1 = EmbedFC(1, 2 * n_feat)
        self.timeembed2 = EmbedFC(1, 1 * n_feat)

        self.n_out1 = 2 * n_feat
        self.n_out2 = n_feat

        self.contextembed1 = nn.ModuleList([EmbedFC(self.n_classes[iclass], self.n_out1) for iclass in range(len(self.n_classes))])
        self.contextembed2 = nn.ModuleList([EmbedFC(self.n_classes[iclass], self.n_out2) for iclass in range(len(self.n_classes))])

        n_conv = 7
        self.up0 = nn.Sequential(
            nn.ConvTranspose2d(2 * n_feat, 2 * n_feat, n_conv, n_conv),
            nn.GroupNorm(8, 2 * n_feat),
            nn.ReLU(),
        )

        self.up1 = UnetUp(4 * n_feat, n_feat)
        self.up2 = UnetUp(2 * n_feat, n_feat)
        self.out = nn.Sequential(
            nn.Conv2d(2 * n_feat, n_feat, 3, 1, 1),
            nn.GroupNorm(8, n_feat),
            nn.ReLU(),
            nn.Conv2d(n_feat, self.in_channels, 3, 1, 1),
        )

    def forward(self, x, c, t, context_mask=None):
        x = self.init_conv(x)
        down1 = self.down1(x)
        down2 = self.down2(down1)
        hiddenvec = self.to_vec(down2)

        temb1 = self.timeembed1(t).view(-1, int(self.n_feat), 1, 1)
        temb2 = self.timeembed2(t).view(-1, int(self.n_feat / 2), 1, 1)

        cemb1 = 0
        cemb2 = 0
        for ic in range(len(self.n_classes)):
            tmpc = c[ic]
            if tmpc.dtype == torch.int64:
                tmpc = nn.functional.one_hot(tmpc, num_classes=self.n_classes[ic]).type(torch.float)
            cemb1 += self.contextembed1[ic](tmpc).view(-1, int(self.n_out1 / 1.), 1, 1)
            cemb2 += self.contextembed2[ic](tmpc).view(-1, int(self.n_out2 / 1.), 1, 1)

        up1 = self.up0(hiddenvec)
        up2 = self.up1(cemb1 * up1 + temb1, down2)
        up3 = self.up2(cemb2 * up2 + temb2, down1)
        out = self.out(torch.cat((up3, x), 1))
        return out


# ───────────────────────────────────────────────────────────────
# EXERCISES: See below for Exercise 1 and Exercise 2 implementations
# ───────────────────────────────────────────────────────────────


## 🎯 Exercise 1: Direct Regression Baseline

This exercise asks you to implement a simpler alternative to flow matching: **direct regression**.
Instead of learning a velocity field, the network directly predicts the target image from context labels only.

### The Regression Model

Create a `Regression` class that:
1. Takes an **all-ones image** (not noise) as input
2. Uses fixed time $t=1$ (not random)
3. Directly predicts the image $\hat{x}_0 \approx x_0$ using only context embedding
4. Computes loss: $\mathcal{L} = \|f(c) - x_0\|^2$

This isolates **what the context embedding alone can learn**, without any interpolated image as a hint.

### Why Compare?

- **Flow matching** uses interpolated image + context → velocity
- **Direct regression** uses context alone → image

The accuracy gap shows how much information the interpolated image provides to the network.

In [ ]:
class Regression(nn.Module):
    def __init__(self, model, device):
        super().__init__()
        self.model = model.to(device)
        self.device = device
        self.mse = nn.MSELoss()

    def forward(self, x0, c_list):
        """
        Direct regression mode:
          - Input is an all-ones image (no information from x_0)
          - Time is always t=1
          - Target is x0 itself (the network directly predicts the image from context)

        Loss: L = |model(ones_image, c, t=1) - x0|^2
        """
        batch_size = x0.shape[0]

        # TODO: Create an all-ones image input of the same shape as x0
        x_t = torch.ones_like(x0)

        # TODO: Create a time tensor with all values set to 1 (shape: [batch_size, 1], device: x0.device)
        t_scalar = torch.ones(batch_size, 1, device=x0.device)

        # TODO: Call self.model(x_t, c_list, t_scalar) to predict the image from context
        x_pred = self.model(x_t, c_list, t_scalar)

        # TODO: Return self.mse(x_pred, x0)
        return self.mse(x_pred, x0)

    @torch.no_grad()
    def sample(self, n, c_list, size):
        """
        Single forward pass: predict the image directly from context.
        No iterative sampling or ODE integration needed.
        """
        # TODO: Create an all-ones input tensor of shape (n, *size) on self.device
        x = torch.ones(n, *size, device=self.device)

        # TODO: Create a time tensor of ones with shape (n, 1) on self.device
        t_tensor = torch.ones(n, 1, device=self.device)
        
        # TODO: Return self.model(x, c_list, t_tensor)
        return self.model(x, c_list, t_tensor)


## 🔁 Training Loop

The `training_model` function below provides the standard supervised training loop.
Both `Regression` and `FlowMatching` share it — they differ only in how `forward` computes the loss.


In [ ]:
def training_model(model, train_loader, args, device, model_name="model"):
    """Train a generative model (Regression or FlowMatching) and save checkpoints."""
    model = model.to(device)
    model.train()
    optim = torch.optim.Adam(model.parameters(), lr=args.lrate)
    
    for ep in tqdm(range(args.n_epoch), desc="Training Epochs"):
        model.train()
        losses = []
        
        for x, c in tqdm(train_loader, desc=f"Epoch {ep+1}", leave=False):
            x = x.to(device)
            cl = [c[i].to(device) for i in c]
            
            if ep == 0 and len(losses) == 0:
                grid = vutils.make_grid(x[:10], nrow=5, normalize=True)
                plt.imshow(grid.permute(1, 2, 0).cpu())
                plt.axis('off')
                plt.title(f"{model_name} - Training Batch")
                plt.show()
            
            # TODO: Zero the gradients
            optim.zero_grad()

            # TODO: Compute the loss by calling model(x, cl)
            loss = model(x, cl)

            # TODO: Backpropagate the loss
            loss.backward()
            
            # TODO: Update the model parameters
            optim.step()

            losses.append(loss.item())
        
        tqdm.write(f"Epoch {ep+1}/{args.n_epoch}, Loss: {np.mean(losses):.4f}")
    
    return model


In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim, output_dims):
        super().__init__()
        self.output_fc0 = nn.Linear(input_dim, output_dims[0])
        self.output_fc1 = nn.Linear(input_dim, output_dims[1])
        self.output_fc2 = nn.Linear(input_dim, output_dims[2])

    def forward(self, x):
        batch_size = x.shape[0]
        x = x[:, :3, :, :].reshape(batch_size, -1)
        y_pred = {}
        y_pred[0] = self.output_fc0(x)
        y_pred[1] = self.output_fc1(x)
        y_pred[2] = self.output_fc2(x)
        return y_pred
    

def calc_acc(preds, obs, classifier, nclasses=3):
    y_pred = classifier(preds)
    accs = []
    preds_out = []
    for ii in range(nclasses):
        top_pred = y_pred[ii].argmax(1, keepdim=True).detach().cpu().numpy()
        acc = np.array(top_pred[:, 0] == int(obs[ii]), dtype=np.int64)
        accs.append(acc)
        preds_out.append(top_pred[:, 0])
    return np.array(preds_out), np.array(accs)

probe_input_dim = args.pixel_size * args.pixel_size * 3
classifier_linear = MLP(probe_input_dim, [3, 3, 2, 2])
classifier_linear.load_state_dict(torch.load(f"{CV_PATH_FLOW_MATCHING}/probe.pt", map_location=torch.device(device)))
classifier_linear = classifier_linear.to(device)

def sample_images(model, args, config="000"):
    model.eval()
    tf = transforms.Compose([transforms.Resize((args.pixel_size, args.pixel_size)), transforms.ToTensor()])
    test_dataset = CLEVR_dataset(
        tf, args.n_sample,
        args.dataset,
        configs=config,
        training=False,
        test_size=args.test_size,
    )
    test_loader = DataLoader(test_dataset, batch_size=args.batch_size, shuffle=False, num_workers=1)
    x_real, c_gen = next(iter(test_loader))
    cl = {k: v[:args.n_sample].to(device) for k, v in c_gen.items()}
    gen = model.sample(args.n_sample, cl, (3, args.pixel_size, args.pixel_size))
    return gen

def evaluate_and_save(model, args, device, output_dir, model_name="model"):
    """Evaluate model on seen and unseen configs, save accuracies to JSON."""
    model.eval()
    
    seen_configs = ["001", "100", "010"]
    unseen_configs = ["000", "011", "110", "111", "101"]
    
    accuracies = {"seen": {}, "unseen": {}}
    
    # Evaluate on seen configs
    for config in seen_configs:
        gen = sample_images(model, args, config)
        gen = torch.clamp(gen, 0, 1)
        _, accs = calc_acc(gen, config, classifier_linear, nclasses=3)
        accuracies["seen"][config] = float(accs.mean())
        print(f"  {model_name} seen   {config}: {accuracies['seen'][config]:.3f}")
        

        mean_image = torch.clamp(gen.mean(dim=0), 0, 1)
        vutils.save_image(mean_image.cpu(),
            os.path.join(output_dir, f"mean_image_{config}.png"),
            normalize=False)
    
    # Evaluate on unseen configs
    for config in unseen_configs:
        gen = sample_images(model, args, config)
        gen = torch.clamp(gen, 0, 1)
        _, accs = calc_acc(gen, config, classifier_linear, nclasses=3)
        accuracies["unseen"][config] = float(accs.mean())
        print(f"  {model_name} unseen {config}: {accuracies['unseen'][config]:.3f}")
        
        mean_image = torch.clamp(gen.mean(dim=0), 0, 1)
        vutils.save_image(mean_image.cpu(),
            os.path.join(output_dir, f"mean_image_{config}.png"),
            normalize=False)
    
    # Save accuracies to JSON
    json_path = os.path.join(output_dir, "accuracies.json")
    with open(json_path, 'w') as f:
        json.dump(accuracies, f, indent=2)
    print(f"\nSaved accuracies to {json_path}")
    
    return accuracies




In [ ]:
print("=" * 60)
print("TRAINING REGRESSION MODEL (Exercise 1)")
print("=" * 60)

torch.manual_seed(args.seed)
random.seed(args.seed)
np.random.seed(args.seed)

configs = ["001", "100", "010"]
tf = transforms.Compose([transforms.Resize((args.pixel_size, args.pixel_size)), transforms.ToTensor()])
train_dataset = CLEVR_dataset(tf, args.num_samples, args.dataset, configs=configs, training=True)
train_dataloader = DataLoader(train_dataset, batch_size=args.batch_size, shuffle=True, num_workers=1)

regression_model = Regression(ContextUnet(3, args.n_feat, [2, 3, 1]), device)
regression_model = training_model(regression_model, train_dataloader, args, device, model_name="Regression")
torch.save(regression_model.state_dict(), args.baseline_model_path)
print(f"\nRegression model saved to {args.baseline_model_path}")


> **Hint:** A well-trained regression model should achieve a training loss below **0.001**.

In [ ]:
regression_model = Regression(ContextUnet(3, args.n_feat, [2, 3, 1]), device)
regression_model.load_state_dict(torch.load(args.baseline_model_path, map_location=device))
print(f"\nRegression model loaded from {args.baseline_model_path}")

print("\n" + "=" * 60)
print("EVALUATING REGRESSION MODEL")
print("=" * 60)
acc_regression = evaluate_and_save(regression_model, args, device, args.baseline_output_dir, model_name="Regression")

## 🎯 Exercise 2: Flow Matching with Velocity Fields

Now that you've implemented direct regression, let's move to the full **Flow Matching** approach. Instead of predicting the image directly, we'll learn a **velocity field** that transports noise to data along straight paths.

### The Flow Matching Model

Create a `FlowMatching` class that:
1. **During training**: Samples random $t \sim U(0, 1)$ and interpolates: $x_t = (1-t)x_0 + t\epsilon$
2. **Predicts velocity**: $v_\theta(x_t, c, t)$ that matches $v^* = \epsilon - x_0$
3. **Computes loss**: $\mathcal{L} = \|v_\theta(x_t, c, t) - (\epsilon - x_0)\|^2$
4. **Sampling**: Integrates backward from $t=1$ (noise) to $t=0$ (data) using Euler steps:
   $$x_{t-\Delta t} = x_t - v_\theta(x_t, c, t) \cdot \Delta t$$

### Why This Is Better

- **Linear paths**: Flow matching uses straight-line interpolation (simpler than diffusion chains)
- **Constant velocity**: The target velocity $v^* = \epsilon - x_0$ is the same for all $t$
- **Fewer steps**: Convergence is faster than diffusion models (20–50 steps vs 1000+)
- **Better information**: The interpolated image $x_t$ provides gradual information about $x_0$


In [ ]:
class FlowMatching(nn.Module):
    def __init__(self, model, n_T, device):
        super().__init__()
        self.model = model.to(device)
        self.n_T = n_T
        self.device = device
        self.mse = nn.MSELoss()

    def forward(self, x0, c_list):
        """
        Flow matching training:
          - Sample t ~ Uniform(0,1)
          - Interpolate: x_t = (1-t)*x0 + t*epsilon
          - Target velocity: v* = epsilon - x0 (constant along the path)
          - Loss: L = |model(x_t, c, t) - v*|^2
        """
        batch_size = x0.shape[0]

        # TODO: Sample epsilon from a standard normal distribution with the same shape as x0
        epsilon = ...

        # TODO: Sample random t ~ Uniform(0,1) with shape (batch_size, 1, 1, 1) on x0.device
        t = ...

        # TODO: Compute the linearly interpolated image: x_t = (1-t)*x0 + t*epsilon
        x_t = ...
        t_scalar = t.view(batch_size, 1)

        # TODO: Compute the target (constant) velocity: v* = epsilon - x0
        v_target = ...

        # TODO: Predict velocity using self.model(x_t, c_list, t_scalar)
        v_pred = ...

        # TODO: Return self.mse(v_pred, v_target)
        raise NotImplementedError("Implement the FlowMatching forward pass")

    @torch.no_grad()
    def sample(self, n, c_list, size):
        """
        Euler backward integration from t=1 (noise) to t=0 (data).
        """
        # TODO: Initialise x as pure Gaussian noise of shape (n, *size) on self.device
        x = ...

        # TODO: Compute the Euler step size: dt = 1.0 / self.n_T
        dt = ...

        for t_val in torch.linspace(1.0, dt, self.n_T):
            # TODO: Build a time tensor of shape (n, 1) filled with t_val on self.device
            t_tensor = ...

            # TODO: Predict velocity at the current time step
            v = ...

            # TODO: Euler step: x = x - v * dt
            x = ...

        return x


In [ ]:
print("\n" + "=" * 60)
print("TRAINING FLOW MATCHING MODEL (Exercise 2)")
print("=" * 60)

torch.manual_seed(args.seed)
random.seed(args.seed)
np.random.seed(args.seed)

flow_model = FlowMatching(ContextUnet(3, args.n_feat, [2, 3, 1]), n_T=args.n_T, device=device)
flow_model = training_model(flow_model, train_dataloader, args, device, model_name="FlowMatching")
torch.save(flow_model.state_dict(), args.save_model_path)
print(f"\nFlow matching model saved to {args.save_model_path}")


> **Hint:** A well-trained flow matching model should achieve a training loss below **0.03**.

In [ ]:
flow_model = FlowMatching(ContextUnet(3, args.n_feat, [2, 3, 1]), n_T=args.n_T, device=device)
flow_model.load_state_dict(torch.load(args.save_model_path, map_location=device))
print(f"\nFlow matching model loaded from {args.save_model_path}")

print("\n" + "=" * 60)
print("EVALUATING FLOW MATCHING MODEL")
print("=" * 60)
acc_flow = evaluate_and_save(flow_model, args, device, args.output_dir, model_name="FlowMatching")